# YOLOv8L 씬별 객체 탐지 → 스크립트화

**목표:** hyuck1.mp4 영상을 씬 분할 후, 튜닝된 YOLOv8L로 각 씬의 객체를 탐지하여 텍스트로 출력

| 항목 | 설정 |
|------|------|
| 모델 | YOLOv8L (yolov8l.pt) |
| 파라미터 | conf=0.25, iou=0.7, imgsz=800 |
| 클래스 | MVP 15 |
| 입력 | hyuck1.mp4 (씬 분할 후 각 씬) |
| 출력 | 씬별 탐지 객체 리스트 (CSV + 텍스트) |

## 0. 환경 설정

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q ultralytics scenedetect[opencv]
print("설치 완료!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE = "/content/drive/MyDrive/SampleVideo.Test"
VIDEO_PATH = os.path.join(BASE, "hyuck1.mp4")
SCENE_DIR = "/content/hyuck1_scenes"

print(f"영상 존재: {os.path.exists(VIDEO_PATH)}")
print(f"영상 크기: {os.path.getsize(VIDEO_PATH)/1024/1024:.1f}MB")

## 1. 씬 분할 (scenedetect + ffmpeg)

In [ ]:
from scenedetect import open_video, SceneManager
from scenedetect.detectors import ContentDetector
import subprocess

# 씬 탐지
video = open_video(VIDEO_PATH)
scene_manager = SceneManager()
scene_manager.add_detector(ContentDetector(threshold=27))  # 기본값
scene_manager.detect_scenes(video)
scene_list = scene_manager.get_scene_list()

print(f"탐지된 씬 수: {len(scene_list)}개")

# 씬별 클립 추출
os.makedirs(SCENE_DIR, exist_ok=True)

for i, (start, end) in enumerate(scene_list):
    start_time = start.get_timecode()
    end_time = end.get_timecode()
    output_path = os.path.join(SCENE_DIR, f"scene_{i+1:04d}.mp4")
    
    subprocess.run([
        'ffmpeg', '-y', '-i', VIDEO_PATH,
        '-ss', start_time, '-to', end_time,
        '-c:v', 'libx264', '-preset', 'fast',
        '-an', output_path
    ], capture_output=True)

clips = sorted([f for f in os.listdir(SCENE_DIR) if f.endswith('.mp4')])
print(f"생성된 클립 수: {len(clips)}개")

## 2. YOLOv8L 씬별 객체 탐지

In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO
from tqdm.auto import tqdm
from collections import Counter

# 튜닝된 파라미터
CONF = 0.25
IOU = 0.7
IMGSZ = 800

# MVP 15 클래스
MVP_CLASS_IDS = [16, 26, 39, 41, 45, 53, 56, 57, 59, 60, 62, 63, 65, 67, 72]

# COCO 클래스명 매핑
COCO_NAMES = {
    16: 'dog', 26: 'handbag', 39: 'bottle', 41: 'cup', 45: 'bowl',
    53: 'apple', 56: 'chair', 57: 'couch', 59: 'bed', 60: 'dining table',
    62: 'tv', 63: 'laptop', 65: 'remote', 67: 'cell phone', 72: 'refrigerator'
}

# 모델 로드
model = YOLO('yolov8l.pt')
print("모델 로드 완료!")

# 씬별 객체 탐지
results_list = []

for clip_name in tqdm(clips, desc="씬별 객체 탐지"):
    clip_path = os.path.join(SCENE_DIR, clip_name)
    scene_num = int(clip_name.split('_')[1].split('.')[0])
    
    # 영상에서 프레임 추출 (중간 프레임 3장 샘플링)
    cap = cv2.VideoCapture(clip_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        cap.release()
        continue
    
    # 프레임 위치: 25%, 50%, 75%
    sample_positions = [int(total_frames * p) for p in [0.25, 0.5, 0.75]]
    
    all_objects = Counter()
    all_details = []  # (클래스명, conf) 상세 기록
    
    for pos in sample_positions:
        cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
        ret, frame = cap.read()
        if not ret:
            continue
        
        # YOLO 추론
        preds = model.predict(
            frame, conf=CONF, iou=IOU, imgsz=IMGSZ,
            classes=MVP_CLASS_IDS, verbose=False
        )
        
        for r in preds:
            for box in r.boxes:
                cls_id = int(box.cls[0])
                conf_val = float(box.conf[0])
                cls_name = COCO_NAMES.get(cls_id, f'class_{cls_id}')
                all_objects[cls_name] += 1
                all_details.append((cls_name, conf_val))
    
    cap.release()
    
    # 고유 객체 리스트 (중복 제거, 빈도순)
    unique_objects = [obj for obj, _ in all_objects.most_common()]
    objects_str = ", ".join(unique_objects) if unique_objects else "(탐지 없음)"
    
    # 상세 정보 (객체명 + 개수)
    detail_str = ", ".join([f"{obj}({cnt})" for obj, cnt in all_objects.most_common()]) if all_objects else "(탐지 없음)"
    
    results_list.append({
        'scene_num': scene_num,
        'clip_name': clip_name,
        'detected_objects': objects_str,
        'detected_objects_detail': detail_str,
        'total_detections': sum(all_objects.values()),
        'unique_classes': len(unique_objects)
    })

df_objects = pd.DataFrame(results_list)
print(f"\n탐지 완료! 총 {len(df_objects)}개 씬")
print(f"객체 탐지된 씬: {len(df_objects[df_objects['total_detections'] > 0])}개")
print(f"객체 없는 씬: {len(df_objects[df_objects['total_detections'] == 0])}개")

## 3. 결과 확인

In [ ]:
from IPython.display import display

# 전체 결과
print("=" * 60)
print("  씬별 탐지 객체 요약")
print("=" * 60)
display(df_objects[['scene_num', 'detected_objects', 'total_detections']].head(20))

# 전체 객체 빈도 통계
print("\n" + "=" * 60)
print("  전체 객체 빈도 (hyuck1.mp4 전체)")
print("=" * 60)

total_counter = Counter()
for _, row in df_objects.iterrows():
    if row['detected_objects'] != '(탐지 없음)':
        for obj in row['detected_objects'].split(', '):
            total_counter[obj] += 1

for obj, cnt in total_counter.most_common():
    print(f"  {obj}: {cnt}개 씬에서 탐지")

## 4. Google Vision 프롬프트용 스크립트 생성

In [ ]:
# 씬별 프롬프트 템플릿 생성
print("=" * 60)
print("  Google Vision 프롬프트에 삽입할 객체 정보")
print("=" * 60)

for _, row in df_objects.iterrows():
    print(f"\n--- Scene {row['scene_num']} ---")
    print(f"[화면 속 탐지된 객체]")
    print(f"{row['detected_objects']}")

## 5. CSV 저장 & Drive 백업

In [ ]:
import shutil

# CSV 저장
csv_path = 'hyuck1_scene_objects.csv'
df_objects.to_csv(csv_path, index=False, encoding='utf-8-sig')

# Drive 백업
drive_path = os.path.join(BASE, csv_path)
shutil.copy(csv_path, drive_path)
print(f"CSV 저장 완료: {drive_path}")

# 다운로드
from google.colab import files
files.download(csv_path)
print("다운로드 시작!")